# Set up

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import random
from tqdm.auto import tqdm
import time
from torch.amp import autocast, GradScaler
import os
import itertools
import optuna
from pathlib import Path
import copy
from contextlib import nullcontext
import joblib
import gc
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

from models import AgeGuidedAutoencoder, AgeGuidedLoss
from metrics import calc_r2_corr

In [ ]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Config

In [ ]:
# Directories setup
data_dir = '/scratch/bng/cartbind/data/UKB_new_data/combined_data_no_outliers'
regions_dir = '/scratch/bng/cartbind/code/MIND_models/region_names'
save_dir = Path('/scratch/bng/cartbind/code/MIND_models/QuantNets/autoencoders/optuna_results_new_apr24')
save_dir.mkdir(parents=True, exist_ok=True)

brain_data_configs = {
    'FC25':  {'data_path': f'{data_dir}/combined_data_master_no_outliers.csv', 'regions_path': f'{regions_dir}/FC25_regions.txt'},
    'FC100': {'data_path': f'{data_dir}/combined_data_master_no_outliers.csv', 'regions_path': f'{regions_dir}/FC100_regions.txt'},
    'MIND':  {'data_path': f'{data_dir}/combined_data_master_no_outliers.csv', 'regions_path': f'{regions_dir}/MIND_regions.txt'}
}

# The target architectures for each configuration
experiments = [
    # 1 Hidden Dim Arc
    ('FC100', '1_hide', [[1024]], 512),
    ('MIND',  '1_hide', [[1024]], 512),
    ('FC25',  '1_hide', [[128]],  64),
    
    # # 2 Hidden Dims Arc (halving latent)
    # ('FC100', '2_hide', [[1024, 512], [1024], [512]], 256),
    # ('MIND',  '2_hide', [[1024, 512], [1024], [512]], 256),
    # ('FC25',  '2_hide', [[128, 64], [128], [64]], 32),
]

N_TRIALS = 300
EARLY_STOP_PATIENCE = 15
NUM_EPOCHS = 150

# Age-guided autoencoder training setup

In [ ]:
def prepare_data(data_path, regions_path, age_column='p21003_i2', device='cuda'):
    if isinstance(device, str):
        device = torch.device(device)
        
    # Load data
    df = pd.read_csv(data_path, index_col=0)
    with open(regions_path, 'r') as f:
        brain_regions = [line.strip() for line in f.readlines()]
    
    X = df[brain_regions].values
    y = df[age_column].values

    X_train, X_temp, age_train, age_temp = train_test_split(X, y, test_size=0.30, random_state=seed)
    X_val, X_test, age_val, age_test = train_test_split(X_temp, age_temp, test_size=0.50, random_state=seed)

    X_scaler = StandardScaler()
    X_train_scaled = X_scaler.fit_transform(X_train)
    X_val_scaled = X_scaler.transform(X_val)
    X_test_scaled = X_scaler.transform(X_test)

    age_scaler = StandardScaler()
    age_train_scaled = age_scaler.fit_transform(age_train.reshape(-1, 1)).flatten()
    age_val_scaled = age_scaler.transform(age_val.reshape(-1, 1)).flatten()
    age_test_scaled = age_scaler.transform(age_test.reshape(-1, 1)).flatten()

    return {
        'train_x': torch.tensor(X_train_scaled, dtype=torch.float32).to(device),
        'train_y': torch.tensor(age_train_scaled, dtype=torch.float32).to(device),
        'val_x': torch.tensor(X_val_scaled, dtype=torch.float32).to(device),
        'val_y': torch.tensor(age_val_scaled, dtype=torch.float32).to(device),
        'test_x': torch.tensor(X_test_scaled, dtype=torch.float32).to(device),
        'test_y': torch.tensor(age_test_scaled, dtype=torch.float32).to(device),
        'input_dim': X_train_scaled.shape[1],
        'age_scaler': age_scaler,
        'X_scaler': X_scaler
    }

In [ ]:
def evaluate_loader(loader, model, scaler, device, X_scaler, age_scaler):
    outputs, targets, age_preds, age_targets = [], [], [], []
    cm = autocast(device.type) if scaler else nullcontext()
    
    with torch.no_grad(), cm:
        for batch_x, batch_age in loader:
            x_hat, _, age_pred = model(batch_x)
            
            outputs.append(x_hat.cpu().numpy())
            targets.append(batch_x.cpu().numpy())
            age_preds.append(age_pred.cpu().numpy())
            age_targets.append(batch_age.cpu().numpy())
            
    out = np.concatenate(outputs)
    target = np.concatenate(targets)
    age_pred = np.concatenate(age_preds)
    age_target = np.concatenate(age_targets)
    
    # Apply Inverse Transforms mapping back to original space
    out_inv = X_scaler.inverse_transform(out)
    target_inv = X_scaler.inverse_transform(target)
    age_pred_inv = age_scaler.inverse_transform(age_pred.reshape(-1, 1)).flatten()
    age_target_inv  = age_scaler.inverse_transform(age_target.reshape(-1, 1)).flatten()

    return target_inv, out_inv, age_target_inv, age_pred_inv

In [ ]:
def objective(trial, data, input_dim, hidden_dims_choices, latent_dim, device, patience=15, num_epochs=150):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if device.type == 'cuda':
        torch.cuda.manual_seed_all(seed)

    if isinstance(device, str):
        device = torch.device(device)

    # Hyperparameter Sampling
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    ae_weight_decay = trial.suggest_float("ae_weight_decay", 1e-7, 1e1, log=True)
    age_weight_decay = trial.suggest_float("age_weight_decay", 1e-7, 1e1, log=True)
    age_predictor_dropout = trial.suggest_float("age_predictor_dropout", 0.0, 0.95, step=0.05)
    hd_str_map = {str(hd): hd for hd in hidden_dims_choices}
    hd_choice_str = trial.suggest_categorical("hidden_dims", list(hd_str_map.keys()))
    hidden_dims = hd_str_map[hd_choice_str]

    # Dynamically scale age weight via Optuna
    recon_weight = trial.suggest_float("recon_weight", 0.01, 0.99, step=0.01)

    age_pred_depth = trial.suggest_int("age_pred_depth", 1, 4)
    age_predictor_hidden_dims = []
    
    # Base the upper bound off the autoencoder's latent dimension
    current_upper_bound = int(np.log2(latent_dim)) - 1
    
    for i in range(age_pred_depth):
        # The lowest possible power we can pick to still have room for the remaining layers
        min_possible = age_pred_depth - i 
        lower_bound = max(1, min_possible)
        
        power = trial.suggest_int(f"age_dim_exp_l{i}", lower_bound, current_upper_bound)
        age_predictor_hidden_dims.append(2 ** power)
        current_upper_bound = power - 1

    try:
        # Initialization
        model = AgeGuidedAutoencoder(
            input_dim=input_dim, latent_dim=latent_dim, hidden_dims=hidden_dims,
            age_predictor_hidden_dims=age_predictor_hidden_dims, 
            age_predictor_dropout=age_predictor_dropout
        ).to(device)
        
        # Use Optuna's suggested weight
        criterion = AgeGuidedLoss(recon_weight=recon_weight, age_weight=1.0 - recon_weight)

        ae_decay, age_decay, no_decay = [], [], []
        for name, param in model.named_parameters():
            if 'bn' in name or 'bias' in name:
                no_decay.append(param)
            elif 'age_predictor' in name:
                age_decay.append(param)
            else:
                ae_decay.append(param)

        optimizer = torch.optim.AdamW([
            {'params': ae_decay, 'weight_decay': ae_weight_decay},
            {'params': age_decay, 'weight_decay': age_weight_decay},
            {'params': no_decay, 'weight_decay': 0.0}
        ], lr=lr)

        # LR Scheduler mapped to total num_epochs
        warmup_epochs = max(1, int(num_epochs * 0.05))
        warmup_scheduler = LinearLR(optimizer, start_factor=0.01, total_iters=warmup_epochs)
        decay_epochs = num_epochs - warmup_epochs
        cosine_scheduler = CosineAnnealingLR(optimizer, T_max=decay_epochs)
        scheduler = SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs])
        
        scaler = GradScaler('cuda') if device.type == 'cuda' else None

        batch_size = 256
        train_loader = DataLoader(TensorDataset(data['train_x'], data['train_y']), batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(TensorDataset(data['val_x'], data['val_y']), batch_size=batch_size, shuffle=False)
        test_loader = DataLoader(TensorDataset(data['test_x'], data['test_y']), batch_size=batch_size, shuffle=False)
        
        age_scaler = data['age_scaler']
        X_scaler = data['X_scaler']

        best_val_loss = float('inf')
        early_stop_counter = 0
        final_metrics = {}
        best_model_state = None

        # Training Loop with Early Stopping via VAL set
        for epoch in range(num_epochs):
            model.train()
            for batch_x, batch_age in train_loader:
                optimizer.zero_grad()
                
                if scaler:
                    with autocast(device.type):
                        x_hat, _, age_pred = model(batch_x)
                        losses = criterion(batch_x, x_hat, batch_age, age_pred)
                    scaler.scale(losses['total_loss']).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    x_hat, _, age_pred = model(batch_x)
                    losses = criterion(batch_x, x_hat, batch_age, age_pred)
                    losses['total_loss'].backward()
                    optimizer.step()
                    
            scheduler.step()

            # Validation Loop (Early Stopping Only)
            model.eval()
            val_total_loss = 0.0

            with torch.no_grad():
                for batch_x, batch_age in val_loader:                
                    # Support autocast context cleanly during validation
                    cm = autocast(device.type) if scaler else nullcontext()
                    with cm:
                        x_hat, _, age_pred = model(batch_x)
                        losses = criterion(batch_x, x_hat, batch_age, age_pred)
                    
                    val_total_loss += losses['total_loss'].item() * batch_x.size(0)
            
            val_total_loss /= len(val_loader.dataset)
            
            # Prune Trials that turn into NaN or Inf
            if np.isnan(val_total_loss) or np.isinf(val_total_loss):
                raise optuna.exceptions.TrialPruned()

            # Early Stopping check
            if val_total_loss < best_val_loss:
                best_val_loss = val_total_loss
                early_stop_counter = 0
                best_model_state = copy.deepcopy(model.state_dict())
            else:
                early_stop_counter += 1

            if early_stop_counter >= patience or epoch == num_epochs - 1:
                break

        # FINAL METRICS COMPUTATION
        if best_model_state is not None:
            model.load_state_dict(best_model_state)
        
        model.eval() 
        
        # Evaluate Train, Val, and Test
        val_target, val_out, val_age_target, val_age_pred = evaluate_loader(val_loader, model, scaler, device, X_scaler, age_scaler)
        test_target, test_out, test_age_target, test_age_pred = evaluate_loader(test_loader, model, scaler, device, X_scaler, age_scaler)
        train_target, train_out, train_age_target, train_age_pred = evaluate_loader(train_loader, model, scaler, device, X_scaler, age_scaler)
        
        # Store Evaluation Metrics
        metrics_map = {
            'val': (val_target, val_out, val_age_target, val_age_pred),
            'test': (test_target, test_out, test_age_target, test_age_pred),
            'train': (train_target, train_out, train_age_target, train_age_pred)
        }

        for prefix, (target, out, age_target, age_pred) in metrics_map.items():
            final_metrics[f'{prefix}_recon_r2'] = r2_score(target, out)
            final_metrics[f'{prefix}_age_r2'] = r2_score(age_target, age_pred)
            final_metrics[f'{prefix}_recon_r2_corr'] = calc_r2_corr(target, out)
            final_metrics[f'{prefix}_age_r2_corr'] = calc_r2_corr(age_target, age_pred)

        # Save all comprehensive metrics to Optuna study trial
        trial.set_user_attr("epochs_trained", epoch - early_stop_counter)
        trial.set_user_attr("final_arch", str(age_predictor_hidden_dims))
        for k, v in final_metrics.items():
            trial.set_user_attr(k, v)
            
        # Return Validation metrics for optimization
        return final_metrics['val_recon_r2_corr'], final_metrics['val_age_r2_corr']

    finally:
        try:
            del model, optimizer, scheduler, scaler, train_loader, val_loader, test_loader
        except NameError:
            pass
            
        gc.collect()
        if device.type == 'cuda' and torch.cuda.is_available():
            torch.cuda.empty_cache()

# Training loop

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Executing on Object: {device}")

for data_name, arc_name, hidden_dims_choices, latent_dim in experiments:
    print(f"\n=======================================================")
    print(f"Optimizing {data_name} | Arch: {arc_name}")
    print(f"Hidden: {hidden_dims_choices} | Latent: {latent_dim}")
    print(f"=======================================================")
    
    # Prep Data
    paths = brain_data_configs[data_name]
    tensors = prepare_data(paths['data_path'], paths['regions_path'], device=device)
    
    # Optuna Setting (Multi-Objective)
    study_name = f"{data_name}_{arc_name}_autoencoder"
    study = optuna.create_study(
        study_name=study_name,
        directions=['maximize', 'maximize'], # recon_r2, age_r2
        sampler=optuna.samplers.TPESampler(seed=seed)
    )
    
    # Optuna logging disabled to keep console clean, using tqdm wrapper instead
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    
    objective_func = lambda trial: objective(
        trial, tensors, tensors['input_dim'], hidden_dims_choices, latent_dim, device,
        patience=EARLY_STOP_PATIENCE, num_epochs=NUM_EPOCHS
    )
    
    with tqdm(total=N_TRIALS, desc=f"{data_name} {arc_name}") as pbar:
        def update_pbar(_, __):
            pbar.update(1)
        study.optimize(objective_func, n_trials=N_TRIALS, callbacks=[update_pbar])
    
    # Extract robust results Dataframe (from trials and their custom attributes)
    trials_df = pd.DataFrame([
        {
            "trial_id": t.number,
            "recon_r2_obj": t.values[0] if t.values else None,
            "age_r2_obj": t.values[1] if t.values else None,
            **t.params,
            **t.user_attrs
        } for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE
    ])
    
    trials_df.to_csv(save_dir / f"{study_name}_optuna_results.csv", index=False)
    print(f"Saved numerical Results to {save_dir / f'{study_name}_optuna_results.csv'}")

    # ==========================
    # Visualizations & Plotting
    # ==========================
    plots_dir = save_dir / "plots" / study_name
    plots_dir.mkdir(parents=True, exist_ok=True)
    
    try:
        # Plot 1: Pareto Front (Standard multi-objective scatter)
        fig_pareto = optuna.visualization.plot_pareto_front(
            study, 
            target_names=["Recon R²", "Age R²"]
        )
        fig_pareto.write_html(str(plots_dir / "pareto_front.html"))
        fig_pareto.write_image(str(plots_dir / "pareto_front.png"))

        # Plot 2: Optimization History (Need to plot per objective target)
        fig_hist_recon = optuna.visualization.plot_optimization_history(
            study, target=lambda t: t.values[0], target_name="Recon R²"
        )
        fig_hist_recon.write_image(str(plots_dir / "opt_history_recon.png"))
        fig_hist_recon.write_html(str(plots_dir / "opt_history_recon.html"))
        fig_hist_age = optuna.visualization.plot_optimization_history(
            study, target=lambda t: t.values[1], target_name="Age R²"
        )
        fig_hist_age.write_image(str(plots_dir / "opt_history_age.png"))
        fig_hist_age.write_html(str(plots_dir / "opt_history_age.html"))

        # Plot 3: Parallel Coordinates
        best_trials = study.best_trials
        reference_trial = best_trials[0] if best_trials else study.trials[0]
        
        plot_params = [p for p in reference_trial.params.keys() if not p.startswith("age_dim_exp_l")]
        
        fig_parallel_recon = optuna.visualization.plot_parallel_coordinate(
            study, 
            target=lambda t: t.values[0], 
            target_name="Recon R²",
            params=plot_params
        )
        fig_parallel_recon.update_traces(line=dict(reversescale=False))
        fig_parallel_recon.write_image(str(plots_dir / "parallel_coord_recon.png"))
        fig_parallel_recon.write_html(str(plots_dir / "parallel_coord_recon.html"))
        
        fig_parallel_age = optuna.visualization.plot_parallel_coordinate(
            study, 
            target=lambda t: t.values[1], 
            target_name="Age R²",
            params=plot_params
        )
        fig_parallel_age.update_traces(line=dict(reversescale=False))
        fig_parallel_age.write_image(str(plots_dir / "parallel_coord_age.png"))
        fig_parallel_age.write_html(str(plots_dir / "parallel_coord_age.html"))

        # Plot 4: Parameter Importances
        fig_importances_recon = optuna.visualization.plot_param_importances(
            study, target=lambda t: t.values[0], target_name="Recon R²"
        )
        fig_importances_recon.write_image(str(plots_dir / "importances_recon.png"))
        fig_importances_recon.write_html(str(plots_dir / "importances_recon.html"))
        fig_importances_age = optuna.visualization.plot_param_importances(
            study, target=lambda t: t.values[1], target_name="Age R²"
        )
        fig_importances_age.write_image(str(plots_dir / "importances_age.png"))
        fig_importances_age.write_html(str(plots_dir / "importances_age.html"))
        
    except Exception as e:
        print(f"Failed to generate plots for {study_name}: {e}")

print("All tasks complete!")